# Generowanie Obsidian Vault — Katalog Zurada

Tworzy vault z plików MD z linkami `[[wikilink]]` dla grafu Obsidian.

**Struktura:**
- `Produkty/` — 100 stron produktów
- `Powierzchnie/` — dozwolone powierzchnie
- `Odradzane/` — odradzane powierzchnie  
- `Metody/` — metody mycia
- `Certyfikaty/` — certyfikaty i normy
- `pH/` — odczyn pH
- `BHP/` — ostrzeżenia BHP
- `Kategorie/` — kategorie sklepowe
- `_Index.md` — strona główna

In [2]:
import json, re, shutil
import pandas as pd
from pathlib import Path
from collections import defaultdict

BASE_DIR     = Path('.')
PRODUCTS_CSV = BASE_DIR / 'extended_zurada_products_with_metadata_cleaned.csv'
DESC_CSV     = BASE_DIR / 'zurada_canonical_descriptions.csv'
VAULT_DIR    = BASE_DIR / 'zurada_vault'

COL_TO_FOLDER = {
    'dozwolone_powierzchnie': 'Powierzchnie',
    'odradzane_powierzchnie': 'Odradzane',
    'metoda_mycia':           'Metody',
    'zgodnosc_i_certyfikaty': 'Certyfikaty',
    'odczyn_ph':              'pH',
    'ostrzezenia_bhp':        'BHP',
}
COL_LABEL = {
    'dozwolone_powierzchnie': 'Dozwolone powierzchnie',
    'odradzane_powierzchnie': 'Odradzane powierzchnie',
    'metoda_mycia':           'Metody mycia',
    'zgodnosc_i_certyfikaty': 'Certyfikaty',
    'odczyn_ph':              'pH',
    'ostrzezenia_bhp':        'Ostrzeżenia BHP',
}
PROD_FOLDER = 'Produkty'
CAT_FOLDER  = 'Kategorie'

print('Konfiguracja gotowa.')


Konfiguracja gotowa.


In [3]:
def safe(name: str) -> str:
    name = str(name).strip()
    name = re.sub(r'[<>:"/\\|?*\x00-\x1f]', '-', name)
    name = name.strip('-').strip('.')
    return name[:120] or 'unnamed'

def parse_list(val) -> list:
    if pd.isna(val) or str(val).strip() == '':
        return []
    try:
        p = json.loads(val)
        return p if isinstance(p, list) else [str(p)]
    except Exception:
        return [str(val)]

def parse_cats(val) -> list:
    s = str(val) if not pd.isna(val) else ''
    return [c.strip() for c in s.split(',') if c.strip()]

def wikilink(folder: str, value: str) -> str:
    return f'[[{folder}/{safe(value)}|{value}]]'

def write_md(path: Path, content: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding='utf-8')

print('Funkcje pomocnicze gotowe.')


Funkcje pomocnicze gotowe.


In [4]:
df = pd.read_csv(PRODUCTS_CSV)
print(f'Produkty: {len(df)}')

desc_df = pd.read_csv(DESC_CSV)
descriptions = {
    (row['value'], row['column']): row['description']
    for _, row in desc_df.iterrows()
    if pd.notna(row.get('description')) and str(row['description']).strip()
}
print(f'Opisy: {len(descriptions)} wpisow')

if VAULT_DIR.exists():
    shutil.rmtree(VAULT_DIR)
    print('Usunieto stary vault.')
VAULT_DIR.mkdir()
print(f'Vault: {VAULT_DIR.resolve()}')


Produkty: 262
Opisy: 462 wpisow
Vault: /Users/adamzurada/projects/ai-knowledge/rag-product-recommend/lessons/7_prepare_knowledge_map_2/zurada_vault


## Krok 1 — Zbierz powiązania produkt ↔ koncept

In [5]:
# concept_products[(folder, safe_value)] = [product_name, ...]
concept_products = defaultdict(list)

for _, row in df.iterrows():
    pname = row['product_name']
    for col, folder in COL_TO_FOLDER.items():
        for v in parse_list(row.get(col)):
            if v:
                concept_products[(folder, safe(v))].append(pname)
    for cat in parse_cats(row.get('categories', '')):
        concept_products[(CAT_FOLDER, safe(cat))].append(pname)

print(f'Konceptow w grafie: {len(concept_products)}')


Konceptow w grafie: 747


In [6]:
concept_products

defaultdict(list,
            {('Powierzchnie',
              'wszystkie powierzchnie'): ['Zurada Lśniąca Pianka', 'Zurada Naczynia Generic Cleaning', 'Zurada Silnik Moc', 'Zurada Owad Stop'],
             ('Powierzchnie', 'tworzywa sztuczne'): ['Zurada Lśniąca Pianka',
              'Zurada Szklany Blask',
              'Zurada Naczynia Generic Cleaning',
              'Zurada Total Clean',
              'Zurada Gastro Błysk',
              'Zurada Twarda Woda',
              'Zurada Biel Pro',
              'Zurada Turbo Czystość',
              'Zurada Nabłyszcz Pro',
              'Zurada Krystal Błysk',
              'Zurada Czysta Posadzka',
              'Zurada Czysta Moc',
              'Zurada Piankowy Blask',
              'Zurada Alka Czystość',
              'Zurada AlkaMoc',
              'Zurada Kwas Pro',
              'Zurada Neutral Błysk',
              'Zurada Błyszcząca Piana',
              'Zurada Szkło Błysk',
              'Zurada Kryształ',
              'Zura

## Krok 2 — Strony konceptów (Metody, Certyfikaty, pH, Powierzchnie, Odradzane, BHP)

In [7]:
n_concepts = 0
for col, folder in COL_TO_FOLDER.items():
    vals = set()
    for v in df[col].dropna():
        vals.update(parse_list(v))

    for value in vals:
        if not value:
            continue
        desc     = descriptions.get((value, col), '')
        products = concept_products.get((folder, safe(value)), [])
        prod_links = '\n'.join(
            f'- [[{PROD_FOLDER}/{safe(p)}|{p}]]' for p in sorted(products)
        )
        content = (
            f'---\n'
            f'typ: {folder.lower()}\n'
            f'kolumna: {col}\n'
            f'liczba_produktow: {len(products)}\n'
            f'---\n\n'
            f'# {value}\n\n'
            f'{desc if desc else "_Brak opisu._"}\n\n'
            f'## Produkty ({len(products)})\n\n'
            f'{prod_links if prod_links else "_Brak powiązanych produktów._"}\n'
        )
        write_md(VAULT_DIR / folder / f'{safe(value)}.md', content)
        n_concepts += 1

print(f'Koncepty: {n_concepts} plikow')


Koncepty: 462 plikow


## Krok 3 — Strony kategorii

In [8]:
cats_all = set()
for v in df['categories'].dropna():
    cats_all.update(parse_cats(str(v)))

for cat in cats_all:
    if not cat:
        continue
    products  = concept_products.get((CAT_FOLDER, safe(cat)), [])
    prod_links = '\n'.join(
        f'- [[{PROD_FOLDER}/{safe(p)}|{p}]]' for p in sorted(products)
    )
    content = (
        f'---\n'
        f'typ: kategoria\n'
        f'liczba_produktow: {len(products)}\n'
        f'---\n\n'
        f'# {cat}\n\n'
        f'## Produkty ({len(products)})\n\n'
        f'{prod_links if prod_links else "_Brak produktów._"}\n'
    )
    write_md(VAULT_DIR / CAT_FOLDER / f'{safe(cat)}.md', content)

print(f'Kategorie: {len(cats_all)} plikow')


Kategorie: 285 plikow


## Krok 4 — Strony produktów

In [9]:
for _, row in df.iterrows():
    pid   = int(row['product_id'])
    pname = str(row['product_name'])
    title = str(row.get('title', ''))
    short = str(row.get('short_description', ''))
    full  = str(row.get('content_encoded', ''))
    right = str(row.get('right_description', ''))
    ph    = str(row.get('odczyn_ph', '')).strip()

    sections = []

    if ph and ph not in ('nan', ''):
        sections.append(f'**pH:** {wikilink("pH", ph)}')

    for col in ['metoda_mycia', 'zgodnosc_i_certyfikaty']:
        vals = parse_list(row.get(col))
        if vals:
            joined = ' · '.join(wikilink(COL_TO_FOLDER[col], v) for v in vals)
            sections.append(f'**{COL_LABEL[col]}:** {joined}')

    doz = parse_list(row.get('dozwolone_powierzchnie'))
    if doz:
        joined = ' · '.join(wikilink('Powierzchnie', v) for v in doz)
        sections.append(f'**{COL_LABEL["dozwolone_powierzchnie"]}:** {joined}')

    odr = parse_list(row.get('odradzane_powierzchnie'))
    if odr:
        joined = ' · '.join(wikilink('Odradzane', v) for v in odr)
        sections.append(f'**{COL_LABEL["odradzane_powierzchnie"]}:** \u26d4 {joined}')

    bhp = parse_list(row.get('ostrzezenia_bhp'))
    if bhp:
        items = '\n'.join(f'- {wikilink("BHP", v)}' for v in bhp)
        sections.append(f'**{COL_LABEL["ostrzezenia_bhp"]}:**\n{items}')

    cats = parse_cats(str(row.get('categories', '')))
    if cats:
        joined = ' · '.join(wikilink(CAT_FOLDER, c) for c in cats)
        sections.append(f'**Kategorie:** {joined}')

    meta_block = '\n\n'.join(sections)

    yaml_cats   = json.dumps(cats, ensure_ascii=False)
    yaml_metody = json.dumps(parse_list(row.get('metoda_mycia')), ensure_ascii=False)
    yaml_certs  = json.dumps(parse_list(row.get('zgodnosc_i_certyfikaty')), ensure_ascii=False)

    def clean(s):
        return s if s not in ('nan', '') else ''

    content = (
        f'---\n'
        f'product_id: {pid}\n'
        f'title: "{clean(title)}"\n'
        f'ph: "{ph if ph != "nan" else ""}"\n'
        f'metody: {yaml_metody}\n'
        f'certyfikaty: {yaml_certs}\n'
        f'kategorie: {yaml_cats}\n'
        f'---\n\n'
        f'# {pname}\n\n'
        f'> {clean(short)}\n\n'
        f'---\n\n'
        f'{meta_block}\n\n'
        f'---\n\n'
        f'## Opis\n\n{clean(full)}\n\n'
        f'## Właściwości\n\n{clean(right)}\n'
    )
    write_md(VAULT_DIR / PROD_FOLDER / f'{safe(pname)}.md', content)

print(f'Produkty: {len(df)} plikow')


Produkty: 262 plikow


## Krok 5 — Strona główna _Index.md

In [10]:
folder_counts = {}
for folder in [PROD_FOLDER, CAT_FOLDER] + list(COL_TO_FOLDER.values()):
    folder_counts[folder] = len(list((VAULT_DIR / folder).glob('*.md')))

product_list = '\n'.join(
    f'- [[{PROD_FOLDER}/{safe(row["product_name"])}|{row["product_name"]}]]'
    for _, row in df.iterrows()
)

index = (
    '# Katalog Zurada — Mapa Wiedzy\n\n'
    '## Nawigacja\n\n'
    '| Folder | Pliki | Opis |\n'
    '|--------|------:|------|\n'
    f'| [[{PROD_FOLDER}]] | {folder_counts[PROD_FOLDER]} | Produkty czyszczące |\n'
    f'| [[{CAT_FOLDER}]] | {folder_counts[CAT_FOLDER]} | Kategorie sklepowe |\n'
    f'| [[Metody]] | {folder_counts["Metody"]} | Metody mycia |\n'
    f'| [[Certyfikaty]] | {folder_counts["Certyfikaty"]} | Certyfikaty i normy |\n'
    f'| [[pH]] | {folder_counts["pH"]} | Odczyn pH |\n'
    f'| [[Powierzchnie]] | {folder_counts["Powierzchnie"]} | Dozwolone powierzchnie |\n'
    f'| [[Odradzane]] | {folder_counts["Odradzane"]} | Odradzane powierzchnie |\n'
    f'| [[BHP]] | {folder_counts["BHP"]} | Ostrzeżenia BHP |\n\n'
    '## Wszystkie produkty\n\n'
    + product_list + '\n'
)
write_md(VAULT_DIR / '_Index.md', index)

total = sum(1 for _ in VAULT_DIR.rglob('*.md'))
print(f'Vault gotowy: {VAULT_DIR.resolve()}')
print(f'Lacznie {total} plikow .md:')
for folder, count in folder_counts.items():
    print(f'  {folder + "/":20s} {count} plikow')


Vault gotowy: /Users/adamzurada/projects/ai-knowledge/rag-product-recommend/lessons/7_prepare_knowledge_map_2/zurada_vault
Lacznie 1010 plikow .md:
  Produkty/            262 plikow
  Kategorie/           285 plikow
  Powierzchnie/        307 plikow
  Odradzane/           44 plikow
  Metody/              22 plikow
  Certyfikaty/         24 plikow
  pH/                  5 plikow
  BHP/                 60 plikow
